# Feature engineering by day

Build a ticker-by-day panel from the raw StockTwits parquet files. This mirrors `2_a_feature_engineering_by_week.ipynb`, but uses daily aggregation and writes monthly parquet shards under `data/processed_day/` so each output file stays small enough for GitHub.

In [1]:
from __future__ import annotations

import json
import logging
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from data.vocab import Vocabulary

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

PROJECT_ROOT

WindowsPath('c:/stocktwits_2026/StockTwit_WM')

In [2]:
RAW_CANDIDATES = [
    Path(r"C:\stocktwits_2026\parquet\feature_wo_messages"),
    Path(r"C:\stocktwits_2026\parquet\features_wo_messages"),
    PROJECT_ROOT.parent / "parquet" / "feature_wo_messages",
    PROJECT_ROOT.parent / "parquet" / "features_wo_messages",
]

RAW_DIR = next((p for p in RAW_CANDIDATES if p.exists()), RAW_CANDIDATES[0])
OUT_DIR = PROJECT_ROOT / "data" / "processed_day"
MONTHLY_DIR = OUT_DIR / "by_month"
SPLIT_DIR = OUT_DIR / "by_split_month"

START_YEAR = 2008
END_YEAR = 2022
TOP_K = 200
MIN_DAYS = 30

FEATURE_COLS = [
    "log_attention",
    "bullish_rate",
    "bearish_rate",
    "unlabeled_rate",
    "attn_growth",
]

OUT_DIR.mkdir(parents=True, exist_ok=True)
MONTHLY_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw dir:", RAW_DIR)
print("Output dir:", OUT_DIR)

Raw dir: C:\stocktwits_2026\parquet\feature_wo_messages
Output dir: c:\stocktwits_2026\StockTwit_WM\data\processed_day


In [3]:
files = sorted(RAW_DIR.rglob("*.parquet"))
print("Number of raw parquet files:", len(files))
for f in files[:10]:
    print(f)

if not files:
    raise FileNotFoundError(f"No parquet files found under {RAW_DIR}")

Number of raw parquet files: 1234
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.0.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.10.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.15.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.20.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.242.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.25.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.30.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.36.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=10\part.5.parquet
C:\stocktwits_2026\parquet\feature_wo_messages\year=2008\month=11\part.36.parquet


The daily panel has the same raw counts and five model features as the weekly panel, with `day` replacing `week`. `attn_growth` is the ticker's percentage change in `log_attention` versus its previous observed active day.

In [4]:
def build_daily_panel(
    parquet_dir: str | Path,
    start_year: int = 2008,
    end_year: int = 2022,
    top_k: int = 200,
) -> pd.DataFrame:
    parquet_dir = Path(parquet_dir)
    glob = (parquet_dir / "year=*" / "month=*" / "*.parquet").as_posix()

    con = duckdb.connect()
    raw = con.execute(f"""
        WITH base AS (
            SELECT
                UNNEST(regexp_extract_all(symbol_list, '''([^'']+)''', 1)) AS symbol,
                user_id,
                sentiment,
                date_trunc('day', created_at::TIMESTAMP)::DATE AS day
            FROM read_parquet('{glob}', hive_partitioning = true)
            WHERE year >= {start_year} AND year <= {end_year}
              AND symbol_list IS NOT NULL
              AND symbol_list != '[]'
        )
        SELECT
            symbol,
            day,
            COUNT(*)                                               AS msg_count,
            COUNT(DISTINCT user_id)                                AS user_count,
            SUM(CASE WHEN sentiment = 'Bullish' THEN 1 ELSE 0 END) AS bullish_count,
            SUM(CASE WHEN sentiment IS NOT NULL THEN 1 ELSE 0 END) AS labeled_count
        FROM base
        WHERE symbol != ''
        GROUP BY symbol, day
        ORDER BY day, symbol
    """).df()

    print(
        f"[features_day] raw panel: {len(raw):,} rows, "
        f"{raw['symbol'].nunique():,} symbols, {raw['day'].nunique():,} days"
    )

    raw["log_attention"] = np.log1p(raw["msg_count"].astype(float))
    raw["bullish_rate"] = np.where(
        raw["labeled_count"] > 0,
        raw["bullish_count"] / raw["labeled_count"],
        0.0,
    )
    raw["bearish_rate"] = 1.0 - raw["bullish_rate"]
    raw["unlabeled_rate"] = 1.0 - np.where(
        raw["msg_count"] > 0,
        raw["labeled_count"] / raw["msg_count"],
        1.0,
    )

    raw = raw.sort_values(["day", "msg_count"], ascending=[True, False])
    raw["rank"] = raw.groupby("day")["msg_count"].rank(method="first", ascending=False)
    panel = raw[raw["rank"] <= top_k].copy().drop(columns=["rank"])

    panel = panel.sort_values(["symbol", "day"]).reset_index(drop=True)
    panel["prev_log_attn"] = panel.groupby("symbol")["log_attention"].shift(1)
    panel["attn_growth"] = (
        (panel["log_attention"] - panel["prev_log_attn"])
        / (panel["prev_log_attn"].abs() + 1e-6)
    ).clip(-5.0, 5.0)
    panel["attn_growth"] = panel["attn_growth"].fillna(0.0)
    panel = panel.drop(columns=["prev_log_attn"])

    panel = panel.sort_values(["day", "symbol"]).reset_index(drop=True)
    panel["day"] = pd.to_datetime(panel["day"])
    return panel

In [5]:
panel = build_daily_panel(
    parquet_dir=RAW_DIR,
    start_year=START_YEAR,
    end_year=END_YEAR,
    top_k=TOP_K,
)

print("Panel shape:", panel.shape)
print("Days:", panel["day"].nunique())
print("Unique tickers:", panel["symbol"].nunique())

panel.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[features_day] raw panel: 15,022,828 rows, 26,016 symbols, 5,331 days
Panel shape: (1033298, 11)
Days: 5331
Unique tickers: 10216


,symbol,day,msg_count,user_count,bullish_count,labeled_count,log_attention,bullish_rate,bearish_rate,unlabeled_rate,attn_growth
0,AAPL,2008-05-27,2,2,0.0,0.0,1.098612,0.0,1.0,1.0,0.0
1,NES,2008-05-27,1,1,0.0,0.0,0.693147,0.0,1.0,1.0,0.0
2,V,2008-05-27,1,1,0.0,0.0,0.693147,0.0,1.0,1.0,0.0
3,XLE,2008-05-27,1,1,0.0,0.0,0.693147,0.0,1.0,1.0,0.0
4,AAPL,2008-05-28,2,2,0.0,0.0,1.098612,0.0,1.0,1.0,0.0


In [6]:
print(panel.columns.tolist())
print(panel.dtypes)

panel.describe(include="all")

['symbol', 'day', 'msg_count', 'user_count', 'bullish_count', 'labeled_count', 'log_attention', 'bullish_rate', 'bearish_rate', 'unlabeled_rate', 'attn_growth']
symbol                       str
day               datetime64[us]
msg_count                  int64
user_count                 int64
bullish_count            float64
labeled_count            float64
log_attention            float64
bullish_rate             float64
bearish_rate             float64
unlabeled_rate           float64
attn_growth              float64
dtype: object


,symbol,day,msg_count,user_count,bullish_count,labeled_count,log_attention,bullish_rate,bearish_rate,unlabeled_rate,attn_growth
count,1033298,1033298,1.033298e+06,1.033298e+06,1.033298e+06,1.033298e+06,1.033298e+06,1.033298e+06,1.033298e+06,1.033298e+06,1.033298e+06
unique,10216,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,5278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2015-12-03 13:40:33.880643,1.978209e+02,7.799380e+01,7.996829e+01,9.443556e+01,3.840085e+00,5.807617e-01,4.192383e-01,7.378433e-01,7.307338e-02
min,NaN,2008-05-27 00:00:00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,6.931472e-01,0.000000e+00,0.000000e+00,0.000000e+00,-8.731388e-01
25%,NaN,2012-05-23 00:00:00,1.400000e+01,8.000000e+00,0.000000e+00,0.000000e+00,2.708050e+00,0.000000e+00,3.012048e-02,5.405405e-01,-1.274971e-01
50%,NaN,2015-12-05 00:00:00,4.600000e+01,2.600000e+01,7.000000e+00,9.000000e+00,3.850148e+00,7.692308e-01,2.307692e-01,7.637795e-01,0.000000e+00
75%,NaN,2019-06-19 00:00:00,1.430000e+02,7.100000e+01,4.400000e+01,5.400000e+01,4.969813e+00,9.698795e-01,1.000000e+00,1.000000e+00,1.640129e-01
max,NaN,2022-12-31 00:00:00,1.452740e+05,4.014900e+04,9.013300e+04,9.751800e+04,1.188638e+01,1.000000e+00,1.000000e+00,1.000000e+00,5.000000e+00


In [7]:
train_mask = panel["day"] < "2019-01-01"
val_mask = (panel["day"] >= "2019-01-01") & (panel["day"] < "2020-01-01")
test1_mask = (panel["day"] >= "2020-01-01") & (panel["day"] < "2020-07-01")
test2_mask = (panel["day"] >= "2020-10-01") & (panel["day"] < "2021-07-01")

splits = {
    "train": panel[train_mask].copy(),
    "val": panel[val_mask].copy(),
    "test1": panel[test1_mask].copy(),
    "test2": panel[test2_mask].copy(),
}

for name, df in splits.items():
    print(f"{name}: {df.shape}, days={df['day'].nunique()}, symbols={df['symbol'].nunique()}")

train: (741098, 11), days=3870, symbols=7819
val: (73000, 11), days=365, symbols=2213
test1: (36400, 11), days=182, symbols=1554
test2: (54600, 11), days=273, symbols=2185


In [8]:
ticker_counts = splits["train"].groupby("symbol")["day"].nunique()
eligible = ticker_counts[ticker_counts >= MIN_DAYS].index.tolist()

print("Eligible tickers:", len(eligible))

vocab = Vocabulary.build(eligible)
vocab.save(OUT_DIR / "vocab.json")

print("Vocabulary size including PAD:", vocab.size)

Eligible tickers: 2586
[vocab] built: 2586 tickers (indices 1–2586)
[vocab] saved 2586 tickers → c:\stocktwits_2026\StockTwit_WM\data\processed_day\vocab.json
Vocabulary size including PAD: 2587


Monthly shards are the GitHub-friendly outputs. `by_month/` contains the full panel, and `by_split_month/` contains one folder per split with monthly files.

In [9]:
def write_monthly_shards(df: pd.DataFrame, out_dir: Path, prefix: str = "panel") -> list[dict]:
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest = []

    df = df.copy()
    df["year_month"] = df["day"].dt.strftime("%Y_%m")

    for year_month, month_df in df.groupby("year_month", sort=True):
        month_df = month_df.drop(columns=["year_month"]).sort_values(["day", "symbol"])
        path = out_dir / f"{prefix}_{year_month}.parquet"
        month_df.to_parquet(path, index=False)
        manifest.append(
            {
                "file": str(path.relative_to(OUT_DIR)).replace("\\", "/"),
                "rows": int(len(month_df)),
                "days": int(month_df["day"].nunique()),
                "symbols": int(month_df["symbol"].nunique()),
                "size_bytes": int(path.stat().st_size),
            }
        )

    return manifest


manifest = {"all": write_monthly_shards(panel, MONTHLY_DIR, "panel")}

for split_name, split_df in splits.items():
    manifest[split_name] = write_monthly_shards(
        split_df,
        SPLIT_DIR / split_name,
        f"panel_{split_name}",
    )

with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

total_files = sum(len(v) for v in manifest.values())
max_file = max(item["size_bytes"] for files in manifest.values() for item in files)
print(f"Saved {total_files} monthly parquet shards under {OUT_DIR}")
print(f"Largest shard: {max_file / 1024 / 1024:.2f} MiB")

Saved 331 monthly parquet shards under c:\stocktwits_2026\StockTwit_WM\data\processed_day
Largest shard: 0.27 MiB


In [10]:
stats = {
    "aggregation": "day",
    "start_year": START_YEAR,
    "end_year": END_YEAR,
    "top_k": TOP_K,
    "min_days": MIN_DAYS,
    "vocab_size_excluding_pad": len(vocab),
    "vocab_size_including_pad": vocab.size,
    "n_rows": len(panel),
    "n_days": panel["day"].nunique(),
    "n_symbols": panel["symbol"].nunique(),
    "feature_cols": FEATURE_COLS,
}

for split_name, split_df in splits.items():
    stats[f"n_{split_name}_rows"] = len(split_df)
    stats[f"{split_name}_days"] = split_df["day"].nunique()
    stats[f"{split_name}_symbols"] = split_df["symbol"].nunique()

with open(OUT_DIR / "dataset_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

stats

{'aggregation': 'day',
 'start_year': 2008,
 'end_year': 2022,
 'top_k': 200,
 'min_days': 30,
 'vocab_size_excluding_pad': 2586,
 'vocab_size_including_pad': 2587,
 'n_rows': 1033298,
 'n_days': 5331,
 'n_symbols': 10216,
 'feature_cols': ['log_attention',
  'bullish_rate',
  'bearish_rate',
  'unlabeled_rate',
  'attn_growth'],
 'n_train_rows': 741098,
 'train_days': 3870,
 'train_symbols': 7819,
 'n_val_rows': 73000,
 'val_days': 365,
 'val_symbols': 2213,
 'n_test1_rows': 36400,
 'test1_days': 182,
 'test1_symbols': 1554,
 'n_test2_rows': 54600,
 'test2_days': 273,
 'test2_symbols': 2185}

In [11]:
first_shard = OUT_DIR / manifest["all"][0]["file"]
pd.read_parquet(first_shard).head()

,symbol,day,msg_count,user_count,bullish_count,labeled_count,log_attention,bullish_rate,bearish_rate,unlabeled_rate,attn_growth
0,AAPL,2008-05-27,2,2,0.0,0.0,1.098612,0.0,1.0,1.0,0.0
1,NES,2008-05-27,1,1,0.0,0.0,0.693147,0.0,1.0,1.0,0.0
2,V,2008-05-27,1,1,0.0,0.0,0.693147,0.0,1.0,1.0,0.0
3,XLE,2008-05-27,1,1,0.0,0.0,0.693147,0.0,1.0,1.0,0.0
4,AAPL,2008-05-28,2,2,0.0,0.0,1.098612,0.0,1.0,1.0,0.0
